# Deep Learning Lab 0

## Imports

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
from torch.utils.data import random_split

## Preprocessing
### Checklist Train Data
- Normalization
- Data Augmentation
- Create images --> ToTensor()
- Dataloader
### Checklist Test Data
- Normalization
- Create images --> ToTensor()

In [ ]:
"""
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4), # Randomly Crops 32x32 Region After Padding To Create Small Shifts --> Robustness
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), 
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
    ])

"""

transformer = transforms.Compose([
    transforms.ToTensor(), 
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
    ])

traindata = torchvision.datasets.CIFAR10(
    root='./data', 
    train=True, 
    download=True, 
    transform=transformer)

testdata = torchvision.datasets.CIFAR10(
    root='./data', 
    train=False, 
    download=True, 
    transform=transformer)

Files already downloaded and verified
Files already downloaded and verified


In [4]:
train_size = int(0.8*len(traindata))
val_size = len(traindata) - train_size

train_dataset, val_dataset = random_split(traindata, [train_size, val_size], generator=torch.Generator().manual_seed(42))

trainload = torch.utils.data.DataLoader(traindata, batch_size=32, shuffle=True)
valload = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)
testload = torch.utils.data.DataLoader(testdata, batch_size=32, shuffle=False)

## Training and Testing 
### Checklist
- Activation Function LeakyReLU
- Optimizer Stochastic Gradient Descent lr = 0.0001
- Accuracy on Test set
- Val/Train Stopping when the Val loss converges
- Tensorboard and wandb
- Schedulers


In [5]:
if torch.cuda.is_available():
    device = "cuda" #Nvidia Graphics Card
elif torch.backends.mps.is_available():
    device = "mps" # Apple
else:
    device = "cpu" # Worst Case
print(device)


cuda


In [6]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # Images: 32x32
            nn.BatchNorm2d(32),
            nn.LeakyReLU(),
            nn.MaxPool2d(2,2), # Images: (32x32)/(2*2) --> 16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(),
            nn.MaxPool2d(2,2) # Images  (16x16)/(2x2) --> 8x8
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 128), # 4096 --> 128
            nn.LeakyReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [22]:
model = CNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)
num_epochs = 800

In [23]:
# ------------------------- Wandb -------------------------


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode="min",
        factor = 0.1, 
        patience = 3 # Wait x amount of epochs before reducing lr
    )

best_val_loss = float('inf')

for epoch in range(num_epochs):
    # ------------------------- TRAIN -------------------------
    model.train()
    running_loss = 0.0

    for images, labels in trainload:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    average_loss = running_loss / len(trainload)
    # ------------------------- VALIDATION -------------------------

    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in valload:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            val_loss = criterion(outputs, labels)
            val_running_loss += val_loss.item()

        average_val_loss = val_running_loss / len(valload)

    scheduler.step(average_loss)

    # ------------------------- Saving Best Model -------------------------
    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        torch.save(model.state_dict(), "BestModel.pth")
        print("Saved The Best Performing Model")

    # ------------------------- Visualization -------------------------
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"train loss: {average_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )

Saved The Best Performing Model
Epoch [1/800] train loss: 2.215, val loss: 2.110, lr: 0.000100
Saved The Best Performing Model
Epoch [2/800] train loss: 2.041, val loss: 1.958, lr: 0.000100
Saved The Best Performing Model
Epoch [3/800] train loss: 1.918, val loss: 1.850, lr: 0.000100
Saved The Best Performing Model
Epoch [4/800] train loss: 1.830, val loss: 1.768, lr: 0.000100
Saved The Best Performing Model
Epoch [5/800] train loss: 1.763, val loss: 1.703, lr: 0.000100
Saved The Best Performing Model
Epoch [6/800] train loss: 1.707, val loss: 1.649, lr: 0.000100
Saved The Best Performing Model
Epoch [7/800] train loss: 1.663, val loss: 1.604, lr: 0.000100
Saved The Best Performing Model
Epoch [8/800] train loss: 1.628, val loss: 1.569, lr: 0.000100
Saved The Best Performing Model
Epoch [9/800] train loss: 1.596, val loss: 1.536, lr: 0.000100
Saved The Best Performing Model
Epoch [10/800] train loss: 1.566, val loss: 1.512, lr: 0.000100
Saved The Best Performing Model
Epoch [11/800] tr

In [24]:
model.load_state_dict(torch.load("BestModel.pth"))
model.to(device)
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testload:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        HighestValue, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

average_test_loss = test_loss / len(testload)
test_acc = 100*(correct/total)

print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")


C:\Users\david\AppData\Local\Temp\ipykernel_24192\1771424678.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("BestModel.pth"))


Test loss: 0.788
Test accuracy: 73.91%


### Checklist 2
- LeakyReLU --> Tanh
- Optimizer: SGD --> Adam
- Visualize the results on a Tensorboard